In [ ]:

# -- Cell 1 -- rclone + Drive.
# Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN attached.
import os, subprocess

r = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
if r.returncode not in (0, 3):
    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")

REMOTE = "drive:Distillation"
out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "cannot see " + REMOTE


In [ ]:

# -- Cell 2 -- deps + GPUs.
#
# THE DEPTH LADDER, measured with the REAL finetuning pipeline.
#
# The linear probe was supposed to choose a truncation depth cheaply. It could
# not: two nuisance parameters each moved it further than the effect being
# measured -- the probe's own regularisation grid by up to 0.081 MCC (3 vs 5 C
# values), and whether the final LayerNorm is kept by up to 0.117 -- against a
# bootstrap CI of +-0.02. Under one grid depth 6 beat the full stack; under the
# other it fell below a bag-of-tokens baseline.
#
# So stop proxying. Run a ladder of depths through their actual LoRA script on
# AmpHGT, which has none of those knobs and is the number we care about. It costs
# about the same GPU time as one more analysis pass.
subprocess.run('pip install -q -U "transformers>=5.0" peft lightning', shell=True, check=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)   # peft/torchao clash guard

import torch, numpy as np, pandas as pd, glob, json, time
print("torch", torch.__version__, "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))
NGPU = max(1, torch.cuda.device_count())


In [ ]:

# -- Cell 3 -- their code and data, our code, both released models.
WORK = "/kaggle/working"
CODE, REPO = WORK + "/distill", WORK + "/project/their_repo"
TEACH = WORK + "/models/peptideclm-2-mlm-large"
SMALL = WORK + "/models/peptideclm-2-mlm-small"
os.makedirs(REPO, exist_ok=True)

def rlsf(path):
    r = subprocess.run("rclone lsf " + path, shell=True, capture_output=True, text=True)
    return r.stdout.split() if r.returncode == 0 else []

def pull_model(name, local):
    """Drive holds the 32M flat and the 337M in HF cache layout."""
    if os.path.exists(local + "/model.safetensors"):
        return "already local"
    flat = "%s/models/%s" % (REMOTE, name)
    if any(f.startswith("model.safetensors") for f in rlsf(flat)):
        os.makedirs(local, exist_ok=True)
        subprocess.run("rclone copy %s %s -P" % (flat, local), shell=True, check=True)
        return "flat"
    snaps = "%s/models/models--aaronfeller--%s/snapshots" % (REMOTE, name)
    shas = [x.rstrip("/") for x in rlsf(snaps)]
    assert shas, "%s not found. Tried  %s  and  %s" % (name, flat, snaps)
    os.makedirs(local, exist_ok=True)
    subprocess.run("rclone copy %s/%s %s -P" % (snaps, shas[0], local),
                   shell=True, check=True)
    return "snapshot " + shas[0][:12]

for sub in ("data", "training"):
    if not os.path.isdir(REPO + "/" + sub):
        subprocess.run("rclone copy %s/their_repo/%s %s/%s --transfers 16 -P"
                       % (REMOTE, sub, REPO, sub), shell=True, check=True)
# Always refresh the code: this notebook depends on the RoPE fix in
# export_truncated.py, and a stale copy already on disk would be used silently.
subprocess.run("rclone copy %s/distill %s --transfers 8 -P" % (REMOTE, CODE),
               shell=True, check=True)

print("teacher <- %s" % pull_model("peptideclm-2-mlm-large", TEACH))
print("small   <- %s" % pull_model("peptideclm-2-mlm-small", SMALL))

TRAIN_PY = REPO + "/training/02_classification_benchmarks_training_code/scripts/classification_finetuning_v2.py"
DATA_DIR = REPO + "/data"

# Our scripts resolve data and models relative to a project root -- they expect
# <root>/their_repo/data and <root>/models/models--aaronfeller--<name>/snapshots/*,
# which is the local checkout's layout. Kaggle's is different, so mirror it with
# symlinks rather than threading a path argument through every call site.
# Omitting this is what made bench_control.py die on an empty glob.
if not os.path.exists(WORK + "/their_repo"):
    os.symlink(REPO, WORK + "/their_repo")
for name, src in (("peptideclm-2-mlm-large", TEACH),
                  ("peptideclm-2-mlm-small", SMALL)):
    d = "%s/models/models--aaronfeller--%s/snapshots/local" % (WORK, name)
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        os.symlink(src, d)

for p in (TRAIN_PY, DATA_DIR + "/amp_train.csv", CODE + "/export_truncated.py",
          WORK + "/their_repo/data/amp_train.csv",
          glob.glob(WORK + "/models/models--aaronfeller--peptideclm-2-mlm-small/snapshots/*")[0]):
    assert os.path.exists(p), "missing: " + p
print("ok -- layout mirrored")


In [ ]:

# -- Cell 4 -- THPep split + the benchmark controls.
#
# bench_control.py writes THPep_train.csv / THPep_test.csv, which their script
# needs and their repo does not ship. It also prints the bag-of-tokens scores that
# every arm below has to clear -- on AmpHGT that bar is MCC ~0.69. An arm that
# does not beat a 405-dim token histogram is not demonstrating anything about its
# representation, however large it is.
r = subprocess.run(["python", "bench_control.py"], cwd=CODE,
                   capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode != 0:
    print("----- STDERR -----"); print(r.stderr[-2000:])
assert r.returncode == 0, "control failed"


In [ ]:

# -- Cell 5 -- build the ladder. CPU only, about a minute each.
#
# Derived here rather than shipped: each rung is a deterministic function of the
# teacher, which is already on Drive, so uploading ~1.6 GB of derived weights and
# pulling them back would cost more than recomputing them. Only a winner is worth
# saving at the end.
#
# export_truncated.py checks each one two ways -- exported block j must be
# bit-identical to source block keep[j] (no forward pass, so nothing can confound
# it), and the loaded model must reproduce the original with those blocks bypassed.
# The second check loads two models in one process, which is the configuration
# that corrupts this family's rotary buffers, so it repairs both first.
EXPORT = WORK + "/compressed"
os.makedirs(EXPORT, exist_ok=True)

LADDER = [8, 16, 24, 31]
made = {}
for d in LADDER:
    out = "%s/peptideclm-2-mlm-trunc%d" % (EXPORT, d)
    if not os.path.exists(out + "/model.safetensors"):
        r = subprocess.run(["python", "export_truncated.py", "--out", out,
                            "--depth", str(d)],
                           cwd=CODE, capture_output=True, text=True)
        print("== trunc%d ==" % d); print(r.stdout[-600:])
        if r.returncode != 0:
            print(r.stderr[-1200:]); continue
    made["trunc%d" % d] = out

# Arms: the two endpoints plus the ladder. The 32M is the bar to beat -- it is
# free and already achieves CellPPD 0.8473 -- and the 337M is the ceiling.
ARMS = {"warmstart32M": SMALL, "full337M": TEACH}
ARMS.update(made)
for k, v in ARMS.items():
    n = json.load(open(v + "/config.json"))["num_blocks"]
    mb = os.path.getsize(v + "/model.safetensors") / 1e6
    print("%-14s %2d blocks  %7.1f MB  %s" % (k, n, mb, v))


In [ ]:

# -- Cell 6 -- run their LoRA script, unmodified, on AmpHGT and THPep.
#
# AmpHGT is the instrument: 9,265 train / 5,148 test, and it has a val file so
# their script takes the single-split branch (1 run per job, not 5-fold). THPep is
# cheap and included as a second reading, but its test set is only 122 molecules
# so treat it as indicative.
#
# CellPPD is deliberately ABSENT. Bag-of-tokens scored 0.8270 there against the
# full encoder's 0.8278 -- it cannot distinguish two backbones, so running it
# would burn GPU to produce a number that means nothing.
#
# --gpu_index must be passed: their Trainer does devices=[int(args.gpu_index)] on
# the raw argument, whose default is None.
# BATCH SIZE IS PER BENCHMARK, and AmpHGT does NOT use their published 32.
#
# AmpHGT molecules are long -- mean 310 tokens, p95 507, max 600 -- and their
# MoleculeDataset passes max_length=None, so nothing is truncated. At batch 32 a
# 16+ block model at d=1024 exhausts a 14.5 GB T4; their published runs used 80 GB
# cards. The first attempt lost trunc16/trunc24/trunc31/full337M to CUDA OOM while
# the two shallowest arms survived -- a depth-ordered failure, not a flaky one.
#
# Every arm of a benchmark must share a batch size or the difference is confounded
# with the depth difference under test, so AmpHGT drops to 16 for ALL arms,
# including the two that already fit. That keeps our AmpHGT numbers internally
# comparable but NOT comparable to their published 0.8844, which used batch 32.
BENCH = {"AmpHGT": 16, "THPep": 32}
SEEDS = [101]
OUT = WORK + "/results/compress_bench"
os.makedirs(OUT, exist_ok=True)

# PULL PREVIOUS RESULTS BACK FIRST. A fresh Kaggle session starts with an empty
# /kaggle/working, so without this the resume check below finds nothing and every
# job re-runs -- including the three-and-a-half hours of THPep that already
# finished.
subprocess.run("rclone copy %s/results/compress_bench %s --transfers 8 -P"
               % (REMOTE, OUT), shell=True, check=False)
done = glob.glob(OUT + "/*/*/seed_*/*_results.csv")
print("recovered %d completed jobs from Drive" % len(done))

# Results are keyed by batch size, so a bs=32 run and a bs=16 run of the same arm
# coexist instead of overwriting each other. The two AmpHGT arms that survived the
# first attempt ran at 32 -- the published setting -- and are the only
# protocol-matched points we have. They are worth keeping alongside the bs=16
# ladder, not replacing with it.
for old_d in glob.glob("%s/*/*/seed_[0-9]*" % OUT):
    if not old_d.rstrip("/").split("seed_")[-1].isdigit():
        continue                       # already migrated
    mig = old_d + "_bs32"              # everything before this change ran at 32
    if not os.path.exists(mig):
        os.rename(old_d, mig)
        print("migrated %s -> %s" % (os.path.basename(old_d), os.path.basename(mig)))

jobs = [(b, a, s) for b in BENCH for a in ARMS for s in SEEDS]
todo = [j for j in jobs
        if not glob.glob("%s/%s/%s/seed_%d_bs%d/*_results.csv"
                         % (OUT, j[0], j[1], j[2], BENCH[j[0]]))]
print("%d jobs, %d to run" % (len(jobs), len(todo)))

bs_used = {}
running, free, t0 = [], list(range(NGPU)), time.time()
while todo or running:
    while todo and free:
        b, arm, s = todo.pop(0)
        gpu = free.pop(0)
        bs = bs_used.get((b, arm), BENCH[b])
        d = "%s/%s/%s/seed_%d_bs%d" % (OUT, b, arm, s, bs)
        os.makedirs(d, exist_ok=True)
        cmd = ["python", TRAIN_PY, "--dataset", b, "--gpu", "0", "--gpu_index", "0",
               "--model_name", ARMS[arm], "--batch_size", str(bs), "--seed", str(s),
               "--data_dir", DATA_DIR, "--save_path", d,
               "--log_dir", "/tmp/logs/%s_%s_%d" % (b, arm, s)]
        p = subprocess.Popen(cmd, cwd=os.path.dirname(TRAIN_PY),
                             stdout=open(d + "/train.log", "w"),
                             stderr=subprocess.STDOUT,
                             env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
        running.append((b, arm, s, gpu, p, d))
        print("[%5.1f min] launch %-8s %-14s gpu%d bs=%d"
              % ((time.time()-t0)/60, b, arm, gpu, bs))
    time.sleep(20)
    for job in list(running):
        b, arm, s, gpu, p, d = job
        if p.poll() is None:
            continue
        running.remove(job); free.append(gpu)
        ok = p.returncode == 0 and glob.glob(d + "/*_results.csv")
        print("[%5.1f min] %-8s %-14s -> %s" % ((time.time()-t0)/60, b, arm,
                                                "ok" if ok else "FAILED rc=%s" % p.returncode))
        if ok:
            subprocess.run("rm -rf /tmp/logs/%s_%s_%d" % (b, arm, s), shell=True)
            continue
        log = open(d + "/train.log").read()
        # Retry an OOM at half the batch rather than losing the arm entirely. The
        # reduced size is recorded so Cell 7 can flag a comparison that is no
        # longer apples-to-apples.
        cur = bs_used.get((b, arm), BENCH[b])
        if "OutOfMemoryError" in log and cur > 4:
            bs_used[(b, arm)] = cur // 2
            todo.append((b, arm, s))
            print("      OOM at bs=%d -> requeued at bs=%d" % (cur, cur // 2))
        else:
            print("".join(log.splitlines(True)[-20:]))
print("")
print("all jobs done in %.1f min" % ((time.time() - t0) / 60))
json.dump({"%s/%s" % k: v for k, v in bs_used.items()},
          open(OUT + "/batch_sizes_reduced.json", "w"), indent=1)
if bs_used:
    print("REDUCED BATCH SIZES -- these arms are NOT comparable to the others:")
    for k, v in sorted(bs_used.items()):
        print("   %s -> %d" % ("/".join(k), v))


In [ ]:

# -- Cell 7 -- the quality-vs-size curve, and ship it.
#
# AmpHGT has a val file so each arm produced ONE prediction set, not five folds,
# so there is nothing to ensemble -- threshold the logits at 0 directly. THPep
# takes the 5-fold branch, so its folds are ensembled by mean logit, the same
# aggregation reverse-engineered for the CellPPD reproduction.
#
# The comparison that matters is against warmstart32M, not against full337M. The
# 32M is free and already good; the teacher is the ceiling. A rung only earns its
# size by beating the 32M.
from sklearn.metrics import matthews_corrcoef, roc_auc_score, accuracy_score

rows = []
for f in sorted(glob.glob(OUT + "/*/*/seed_*/*_results.csv")):
    run = os.path.basename(os.path.dirname(f))          # seed_101_bs16
    seed = int(run.split("_")[1])
    bs = int(run.split("_bs")[1]) if "_bs" in run else 32
    arm = os.path.basename(os.path.dirname(os.path.dirname(f)))
    bench = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(f))))
    d = pd.read_csv(f)
    if "fold" in d.columns and d.fold.nunique() > 1:
        d["i"] = d.groupby("fold").cumcount()
        g = d.groupby("i")
        y, p = g.true_label.first().values, g.predicted_label.mean().values
    else:
        y, p = d.true_label.values, d.predicted_label.values
    nb = json.load(open(ARMS[arm] + "/config.json"))["num_blocks"] if arm in ARMS else -1
    mb = os.path.getsize(ARMS[arm] + "/model.safetensors") / 1e6 if arm in ARMS else 0
    rows.append(dict(bench=bench, arm=arm, seed=seed, bs=bs, blocks=nb,
                     MB=round(mb, 1),
                     mcc=matthews_corrcoef(y, (p > 0).astype(int)),
                     auc=roc_auc_score(y, p),
                     acc=accuracy_score(y, (p > 0).astype(int))))
res = pd.DataFrame(rows).sort_values(["bench", "bs", "MB"])
pd.set_option("display.width", 200)
print(res.to_string(index=False))

# Published PeptideCLM-2 numbers, mean over seeds 101/202/303, from
# figure_generation/results/runs_LoRA_highrank/*_all.csv. Comparable ONLY to arms
# that ran at their batch size of 32 -- a different batch size is a different
# protocol, not a different model.
PUBLISHED = {"AmpHGT": {"mlm-large": 0.8844, "mtr-large": 0.8531,
                        "hybrid-large": 0.8497, "xgboost-morgan": 0.8356,
                        "ChemBERTa-77M": 0.8141},
             "THPep": {"mlm-large": 0.7557, "hybrid-large": 0.7473,
                       "mtr-large": 0.6977, "xgboost-rdkit": 0.6728}}
CONTROL = {"AmpHGT": 0.6862, "THPep": 0.6854}     # bag-of-tokens, from Cell 4

# GROUPED BY BATCH SIZE. Arms trained at different batch sizes cannot be compared
# to each other: the batch difference would be confounded with the depth
# difference under test. AmpHGT needed 16 for the deep arms on a 14.5 GB T4, while
# the two shallowest also have bs=32 runs that ARE comparable to the paper.
for b in res.bench.unique():
    for bs in sorted(res[res.bench == b].bs.unique()):
        sub = res[(res.bench == b) & (res.bs == bs)].sort_values("MB")
        base = sub[sub.arm == "warmstart32M"].mcc
        note = "  [their protocol]" if bs == 32 else "  [OOM-reduced, NOT comparable to published]"
        print("\n=== %s  batch=%d ===%s" % (b, bs, note))
        print("    bag-of-tokens control %.4f" % CONTROL.get(b, float("nan")))
        for _, r in sub.iterrows():
            tag = "  vs 32M %+.4f" % (r.mcc - base.iloc[0]) if len(base) else ""
            flag = "" if r.mcc > CONTROL.get(b, 9) else "   <- BELOW CONTROL"
            print("   %-14s %7.1f MB  MCC %.4f%s%s" % (r.arm, r.MB, r.mcc, tag, flag))
        if bs == 32:
            print("   -- published, same protocol --")
            for k, v in sorted(PUBLISHED.get(b, {}).items(), key=lambda x: -x[1]):
                print("   %-14s %7s     MCC %.4f" % (k, "337M" if "large" in k else "-", v))

res.to_csv(OUT + "/compress_bench_metrics.csv", index=False)
DEST = REMOTE + "/results/compress_bench"
subprocess.run("rclone copy %s %s --drive-chunk-size 64M -P" % (OUT, DEST),
               shell=True, check=True)
print("\nuploaded -> " + DEST)
